# 02 — Latent Trajectories
## Learning the Hidden State of Working Memory via Sequential VAE

---

**The problem PCA doesn't solve:**

In Module 1 you projected data onto PCA axes. PCA finds the directions of maximum *variance* across the entire dataset. But variance is not the same as *cognitive meaning*. The first PC might capture electrode-to-electrode correlation. The second might track respiration.

**What we actually want:** a latent space where the coordinates encode cognitive state — specifically, which letter is being maintained and whether the WM system is stable or about to fail.

**The solution:** Train a model to compress each trial's neural trajectory into a low-dimensional latent trajectory that captures the *dynamics* of WM maintenance. This is exactly what a Sequential VAE does.

---

**Read before this notebook:**
- Kingma & Welling (2013) — *Auto-Encoding Variational Bayes* — **all 8 pages**
- Pandarinath et al. (2018) — LFADS — **methods section only**

If you don't understand the ELBO derivation, go to `00_foundations/00c_probability_inference.ipynb` first.

---
## 1. Why a VAE and Not Just an Autoencoder?

An **autoencoder** compresses data into a latent code and reconstructs it. The latent space is arbitrary — you can't interpolate, you can't sample, you can't interpret the coordinates.

A **VAE** learns a *probabilistic* latent space. Each input maps to a *distribution* over latent codes, not a single point:
$$q(z | x) = \mathcal{N}(\mu(x), \sigma^2(x))$$

The training objective (ELBO) is:
$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{\text{KL}[q(z|x) \| p(z)]}_{\text{regularization}}$$

**Why this matters for neural data:**
1. The regularization ensures the latent space is *smooth* — nearby latent codes correspond to similar neural states
2. The probabilistic encoding naturally represents *uncertainty* — uncertain trials have larger $\sigma^2$
3. The learned latent space can be analyzed geometrically — principal angles, tangling, etc.

**Sequential VAE:** Instead of encoding each time point independently, we use a GRU that processes the entire temporal sequence and outputs a $\mu(t), \sigma(t)$ at each time step. This captures the *trajectory*, not just the instantaneous state.

### ✏️ Exercise — Derive the ELBO

Before running any code:
1. Start from $\log p(x) = \log \int p(x,z) dz$
2. Introduce $q(z|x)$ using the importance sampling trick
3. Apply Jensen's inequality to get the lower bound
4. Show that the bound equals reconstruction - KL

Write the full derivation in `notes/elbo_derivation.md`. If you can't derive it, you will not be able to diagnose why your model fails. **This is not optional.**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../scripts').resolve()))
from ieeg_utils import load_subject, preprocess, high_gamma_power, epoch_data, baseline_normalize, reject_bad_channels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

In [ ]:
# ─── Load and preprocess (same as Module 1) ───────────────────────────────────
# If you already ran Module 1, load the saved epochs instead

subj = 'al'
d    = load_subject(subj)
data, stim, task, target = d['data'], d['stim'], d['task'], d['target']
srate = d['srate']

good_ch  = reject_bad_channels(data)
data_c   = preprocess(data[:, good_ch], srate)
hgp      = high_gamma_power(data_c, srate)
ep_dict  = epoch_data(hgp, stim, task, target, pre_ms=200, post_ms=1500, srate=srate)
epochs   = baseline_normalize(ep_dict['epochs'], ep_dict['times'])
task_id  = ep_dict['task_id']
tgt_id   = ep_dict['tgt_id']

# Convert to PyTorch: (n_trials, n_times, n_channels)
X = torch.tensor(epochs, dtype=torch.float32)
print(f'Input shape: {X.shape}')
print(f'n_trials={X.shape[0]}, n_times={X.shape[1]}, n_channels={X.shape[2]}')

---
## 2. Sequential VAE Architecture

```
Input x(t): [batch, time, channels]
      ↓
  GRU Encoder → h(t): [batch, time, hidden_dim]
      ↓
  μ(t), logσ²(t): [batch, time, latent_dim]   ← Linear layers
      ↓
  Reparameterize: z(t) = μ(t) + ε·σ(t),  ε ~ N(0,I)
      ↓
  GRU Decoder → x̂(t): [batch, time, channels]
      ↓
  Loss = MSE(x, x̂) + β·KL(q||p)
```

**The reparameterization trick is the key innovation of VAEs:** To backpropagate through a *sampling* operation, rewrite $z \sim \mathcal{N}(\mu, \sigma^2)$ as $z = \mu + \sigma \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0,I)$. Now $z$ is a deterministic function of $\mu$ and $\sigma$ (both produced by the encoder), so gradients flow normally.

**KL annealing** ($\beta$ schedule): Start with $\beta = 0$ (pure autoencoder). Increase $\beta$ by 0.02 per epoch until $\beta = 1$. Why: if KL pressure is too strong too early, the encoder collapses to the prior ($\mu = 0, \sigma = 1$) and the latent space carries no information about the input. KL annealing prevents this.

In [ ]:
class GRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, n_layers=1):
        super().__init__()
        self.gru      = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True)
        self.fc_mu    = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        # x: (batch, time, input_dim)
        h, _ = self.gru(x)              # (batch, time, hidden_dim)
        mu    = self.fc_mu(h)           # (batch, time, latent_dim)
        logvar = self.fc_logvar(h)
        return mu, logvar


class GRUDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, n_layers=1):
        super().__init__()
        self.gru = nn.GRU(latent_dim, hidden_dim, n_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        # z: (batch, time, latent_dim)
        h, _ = self.gru(z)
        return self.fc(h)              # (batch, time, output_dim)


class SequentialVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=8):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder = GRUDecoder(latent_dim, hidden_dim, input_dim)
        self.latent_dim = latent_dim

    @staticmethod
    def reparameterize(mu, logvar):
        """z = μ + ε·σ where ε ~ N(0,I). Gradient flows through μ and σ."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z    = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar, z


def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    """
    ELBO = -reconstruction + beta * KL

    Reconstruction: MSE between x and x_hat (assumes Gaussian likelihood)
    KL: closed-form KL between N(mu, sigma^2) and N(0,1)
       = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))

    WHY this form of KL: For q = N(mu, sigma^2) and p = N(0,1),
    the KL divergence has a closed-form solution.
    Derive it in notes/elbo_derivation.md.
    """
    recon = nn.functional.mse_loss(x_hat, x, reduction='mean')
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon.item(), kl.item()


# Instantiate
input_dim  = X.shape[2]   # n_channels
model      = SequentialVAE(input_dim, hidden_dim=64, latent_dim=8).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Input dim: {input_dim}, Latent dim: 8')

In [ ]:
# ─── Training loop ─────────────────────────────────────────────────────────────
# KL annealing: beta = min(epoch * 0.02, 1.0)
# Gradient clipping: max norm = 1.0 (prevents exploding gradients in RNNs)

dataset    = TensorDataset(X)
loader     = DataLoader(dataset, batch_size=16, shuffle=True)
optimizer  = optim.Adam(model.parameters(), lr=1e-3)

n_epochs   = 100
history    = {'total': [], 'recon': [], 'kl': [], 'beta': []}

for epoch in range(n_epochs):
    beta = min(epoch * 0.02, 1.0)   # KL annealing
    model.train()
    epoch_total, epoch_recon, epoch_kl = [], [], []

    for (batch,) in loader:
        batch  = batch.to(device)
        x_hat, mu, logvar, z = model(batch)
        loss, recon, kl      = vae_loss(batch, x_hat, mu, logvar, beta=beta)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()

        epoch_total.append(loss.item())
        epoch_recon.append(recon)
        epoch_kl.append(kl)

    history['total'].append(np.mean(epoch_total))
    history['recon'].append(np.mean(epoch_recon))
    history['kl'].append(np.mean(epoch_kl))
    history['beta'].append(beta)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | beta={beta:.2f} | loss={history["total"][-1]:.4f} | '
              f'recon={history["recon"][-1]:.4f} | kl={history["kl"][-1]:.4f}')

print('Training complete.')

In [ ]:
# ─── Diagnose training ────────────────────────────────────────────────────────
# LEARN TO READ THIS PLOT. It tells you what happened.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['recon'], 'steelblue', lw=1.5)
axes[0].set_title('Reconstruction Loss\n(should decrease monotonically)')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['kl'], 'crimson', lw=1.5)
axes[1].axhline(0.001, color='orange', lw=1, linestyle='--',
                label='Posterior collapse threshold')
axes[1].set_title('KL Divergence\n(near 0 = posterior collapse)')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=9)

axes[2].plot(history['beta'], 'gray', lw=1.5)
axes[2].set_title('Beta (KL weight)\n(KL annealing schedule)')
axes[2].set_xlabel('Epoch')

plt.suptitle('Training Diagnostics — Read These Carefully', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nDiagnosis guide:')
print('  KL ≈ 0 throughout: posterior collapse — encoder ignores input')
print('  Recon stops decreasing early: model underfitting — try larger hidden_dim')
print('  Recon oscillates: learning rate too high — reduce to 1e-4')
print('  KL grows without bound: beta too high too fast — slow the annealing')

In [ ]:
# ─── Extract latent trajectories ──────────────────────────────────────────────
model.eval()
with torch.no_grad():
    _, mu_all, logvar_all, z_all = model(X.to(device))

mu_np = mu_all.cpu().numpy()      # (n_trials, n_times, latent_dim)
z_np  = z_all.cpu().numpy()

print(f'Latent trajectories shape: {mu_np.shape}')

# Reduce to 3D via PCA for visualization
from sklearn.decomposition import PCA
pca3 = PCA(n_components=3)
mu_flat   = mu_np.reshape(-1, mu_np.shape[-1])    # (n_trials*n_times, latent_dim)
scores_3d = pca3.fit_transform(mu_flat).reshape(mu_np.shape[0], mu_np.shape[1], 3)
print(f'Variance explained by top 3 latent PCs: {pca3.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# ─── Visualize latent trajectories ────────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D
from ieeg_utils import TASK_CODES

cond_colors = {0: 'royalblue', 1: 'seagreen', 2: 'crimson'}

fig = plt.figure(figsize=(15, 5))
for col, cond in enumerate([0, 1, 2]):
    ax = fig.add_subplot(1, 3, col+1, projection='3d')
    mask      = task_id == cond
    trial_idx = np.where(mask)[0][:15]  # plot 15 trials
    for ti in trial_idx:
        traj = scores_3d[ti]   # (n_times, 3)
        ax.plot(traj[:,0], traj[:,1], traj[:,2],
                color=cond_colors[cond], lw=0.8, alpha=0.5)
        ax.scatter(*traj[0], color=cond_colors[cond], s=20, alpha=0.7)   # start
    ax.set_title(f'{TASK_CODES[cond]}', fontweight='bold')
    ax.set_xlabel('Latent 1'); ax.set_ylabel('Latent 2'); ax.set_zlabel('Latent 3')

plt.suptitle('Sequential VAE Latent Trajectories — 3D PCA', fontweight='bold')
plt.tight_layout()
plt.show()

# Compare conditions: do they separate?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, dim in enumerate([0, 1, 2]):
    ax = axes[col]
    times_ep = ep_dict['times']
    for cond in [0, 1, 2]:
        mask = task_id == cond
        mean_traj = scores_3d[mask, :, dim].mean(axis=0)
        ax.plot(times_ep, mean_traj, color=cond_colors[cond],
                lw=2, label=TASK_CODES[cond])
    ax.set_title(f'Latent dim {dim+1}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Mean latent coordinate')
    ax.axvline(0, color='gray', lw=1, linestyle='--')
    ax.legend(fontsize=9)

plt.suptitle('Mean Latent Trajectories by Condition', fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✏️ Exercises

### A — Posterior Collapse Stress Test
Retrain with beta immediately = 1.0 (no annealing). What happens to the KL loss? What do the trajectories look like? What does this tell you about the information content of the latent space?

### B — Latent Dimensionality
Try latent_dim = 2, 4, 8, 16. For each: plot the scree plot of the latent PCA and report the reconstruction loss. At what dimensionality does performance saturate? What is the effective dimensionality of WM dynamics?

### C — Decode from Latent
Fit a logistic regression from the mean latent representation (averaged over maintenance window, 0.5-1.5s) to task condition (0/1/2-back). What's the cross-validated accuracy? Do this with the raw high-gamma features as a baseline. Which is better?

### D — The Deep Question
The VAE encodes *uncertainty* in the logvar. Plot the mean logvar across the trial for each condition. Does uncertainty differ between encoding (0-0.5s) vs. maintenance (0.5-1.5s) periods? Between 0-back and 2-back? What would it mean if 2-back maintenance had higher logvar?

---
## Next
Write `notes/reparameterization_trick.md` — derive it from scratch.  
Then: `03_geometric_biomarker/03_principal_angles_tangling.ipynb`